In [36]:
import sys
sys.path.append('..')

import pandas as pd
from utils.db_utils import write_table

In [37]:
def transform_unemp_rate_by_age(file_path):
    df = pd.read_csv(file_path)

    # 1. Pivot dataframe
    df_long = df.melt(
        id_vars=["statistics"],
        var_name="year",
        value_name="value"
    )

    # 2. Parse statistics into age_group and qualification
    def parse_stats(stat_name):
        if "_unemp_rate_" in stat_name:
            parts = stat_name.split("_unemp_rate_", 1)
            age_group = parts[0].strip()
            qualification = parts[1].strip()
            return age_group, qualification
        else:
            return "Total", stat_name.strip()

    df_long[['age_group', 'qualification']] = df_long['statistics'].apply(
        lambda x: pd.Series(parse_stats(x))
    )

    # 3. Convert numeric and scale
    df_long['unemp_rate'] = pd.to_numeric(df_long['value'], errors='coerce')
    df_long = df_long.dropna(subset=['unemp_rate'])

    df_final = df_long[['year', 'age_group', 'qualification', 'unemp_rate']].copy()
    df_final = df_final.sort_values(['year', 'age_group', 'qualification']).reset_index(drop=True)

    return df_final

In [38]:
df_final = transform_unemp_rate_by_age("../../data/Unemployment Rate by Age Group.csv")
df_final.head(20)

,year,age_group,qualification,unemp_rate
0,2016,25 - 34,degree,4.2
1,2016,25 - 34,diploma,3.1
2,2016,35 - 44,degree,1.0
3,2016,35 - 44,diploma,1.6
4,2016,≤ 24,degree,26.9
5,2016,≤ 24,diploma,12.6
6,2016,≥ 45,degree,0.7
7,2016,≥ 45,diploma,0.4
8,2017,25 - 34,degree,3.9
9,2017,25 - 34,diploma,3.6


In [39]:
write_table(df_final, "sc_bronze","dosm_unemp_rate_age")

Table sc_bronze.dosm_unemp_rate_age written successfully.
